# Tobacco Data Gateway — Example Queries

This notebook demonstrates how to use the `tobacco_gateway` package to fetch and explore tobacco-related data for Munich → Bavaria → Germany.

Run from the repo root with the venv active: `.venv/bin/jupyter notebook`

In [1]:
import sys
sys.path.insert(0, '..')  # add repo root to path when running from notebooks/

from tobacco_gateway import fetch, query
import pandas as pd

## 1. Which sources can answer a question?

Use `query()` to find relevant sources before fetching data.

In [2]:
results = query("e-cigarette use trend by age group")
for r in results:
    print(f"{r.score:3d}  {r.source_id:35s}  {r.geographic_level}")

  4  bzga_drogenaffinitaet                germany
  4  eurobarometer_tobacco                germany
  4  muenchen_gesundheitsbericht          munich
  3  bgs_bayern                           bavaria
  3  bzga_rauchverhalten                  germany
  3  debra                                germany
  3  muenchen_gesundheitsbefragung        munich
  3  rki_kiggs                            germany
  2  destatis_mikrozensus                 germany
  2  itc_germany                          germany
  2  rki_geda                             germany
  2  staba_mikrozensus                    bavaria
  1  gesundheitsatlas_bayern              bavaria


In [3]:
# Geographic cascade: prefer Munich, fall back to Bavaria, then Germany
results = query("smoking prevalence Munich")
for r in results:
    print(f"{r.score:3d}  {r.source_id:35s}  {r.geographic_level}")

  8  muenchen_gesundheitsbefragung        munich
  8  muenchen_gesundheitsbericht          munich
  5  gesundheitsatlas_bayern              bavaria
  5  staba_mikrozensus                    bavaria
  4  bgs_bayern                           bavaria
  3  bzga_rauchverhalten                  germany
  3  debra                                germany
  3  rki_geda                             germany
  2  bzga_drogenaffinitaet                germany
  2  destatis_mikrozensus                 germany
  2  eurobarometer_tobacco                germany
  2  itc_germany                          germany
  2  rki_kiggs                            germany


## 2. Germany-level smoking trend (Destatis Mikrozensus)

The Mikrozensus covers 2009–2017 with district-level detail. Freely available, auto-downloaded.

In [4]:
mz = fetch("destatis_mikrozensus")
print(f"Shape: {mz.shape}")
mz.head()

Shape: (51, 16)


,sex,year,age_group,pop_1000,pop_with_data_1000,response_rate_pct,smokers_total_1000,smokers_occasional_1000,smokers_regular_1000,smokers_heavy_1000,nonsmokers_total_1000,former_smokers_1000,avg_smoking_start_age,source_id,geographic_level,location
0,Männlich,2017,15 - 20,2132,1605,75.281426,204,50,154,6.0,1402,18,15.967067,destatis_mikrozensus,germany,Deutschland
1,Männlich,2017,20 - 25,2326,1749,75.193465,521,107,415,24.0,1228,93,16.695943,destatis_mikrozensus,germany,Deutschland
2,Männlich,2017,25 - 30,2768,2111,76.264451,741,132,608,41.0,1370,205,16.785014,destatis_mikrozensus,germany,Deutschland
3,Männlich,2017,30 - 35,2695,2066,76.660482,755,123,632,56.0,1310,317,16.694769,destatis_mikrozensus,germany,Deutschland
4,Männlich,2017,35 - 40,2630,2016,76.653992,726,104,621,68.0,1290,358,16.971660,destatis_mikrozensus,germany,Deutschland


In [5]:
# Smoking prevalence over time (total population, both sexes)
total = mz[mz['sex'] == 'insgesamt'].copy()
total['smoker_pct'] = total['smokers_total_1000'] / total['pop_with_data_1000'] * 100
total[['year', 'smoker_pct']].dropna().sort_values('year')

,year,smoker_pct


## 3. Bavaria-level reports (BGS Bayern)

LGL Bayern publishes health and addiction monitoring reports. PDFs are downloaded to `data/bgs_bayern/`.

In [6]:
bgs = fetch("bgs_bayern")
bgs

,source_id,geographic_level,location,year,pdf_path,url,status
0,bgs_bayern,bavaria,Bayern,2021,/home/felix/tdata/notebooks/../data/bgs_bayern...,https://www.lgl.bayern.de/publikationen/gesund...,cached
1,bgs_bayern,bavaria,Bayern,2014,/home/felix/tdata/notebooks/../data/bgs_bayern...,https://www.lgl.bayern.de/publikationen/doc/ge...,cached


The 2021 Suchtmonitoring Bayern report focuses specifically on smoking behavior.
Open the PDF to extract specific tables using `pdfplumber`:

In [7]:
import pdfplumber

pdf_path = bgs.loc[bgs['year'] == 2021, 'pdf_path'].iloc[0]
with pdfplumber.open(pdf_path) as pdf:
    print(f"Pages: {len(pdf.pages)}")
    for i, page in enumerate(pdf.pages[:5]):
        text = page.extract_text() or ""
        if 'rauchen' in text.lower():
            print(f"  Page {i+1}: contains smoking content")

Pages: 14


  Page 1: contains smoking content


  Page 2: contains smoking content


  Page 3: contains smoking content


  Page 4: contains smoking content


  Page 5: contains smoking content


## 4. Munich-level reports

The Münchner Gesundheitsbericht and Gesundheitsbefragung are the most local sources.

In [8]:
gb = fetch("muenchen_gesundheitsbericht")
gbf = fetch("muenchen_gesundheitsbefragung")
pd.concat([gb, gbf], ignore_index=True)

,source_id,geographic_level,location,year,pdf_path,url,status
0,muenchen_gesundheitsbericht,munich,München,2015,/home/felix/tdata/notebooks/../data/muenchen_g...,https://stadt.muenchen.de/dam/jcr:6bfb400e-740...,cached
1,muenchen_gesundheitsbefragung,munich,München,2016,/home/felix/tdata/notebooks/../data/muenchen_g...,https://stadt.muenchen.de/dam/jcr:f24696d1-9f1...,cached


## 5. Sources requiring manual steps

Some sources require a data-use agreement or manual export. `fetch()` raises a `RuntimeError` with step-by-step instructions.

In [9]:
for src in ['rki_geda', 'staba_mikrozensus', 'rki_kiggs', 'eurobarometer_tobacco']:
    try:
        fetch(src)
        print(f"{src}: OK")
    except (RuntimeError, FileNotFoundError) as e:
        print(f"{src}: manual step needed — {str(e).splitlines()[0]}")

rki_geda: manual step needed — GEDA aggregate tables cannot be scraped from the current RKI website.
staba_mikrozensus: manual step needed — GENESIS Bayern no longer allows anonymous downloads.
rki_kiggs: manual step needed — KiGGS microdata must be downloaded from the RKI FDZ.
eurobarometer_tobacco: manual step needed — Eurobarometer tobacco microdata requires a free GESIS account.


## 6. All sources at a glance

In [10]:
import pathlib, re

rows = []
for source_dir in sorted(pathlib.Path('../sources').iterdir()):
    md = source_dir / 'SOURCE.md'
    if not md.exists():
        continue
    text = md.read_text()
    def field(key):
        m = re.search(rf'^{key}: (.+)$', text, re.MULTILINE)
        return m.group(1).strip('"') if m else ''
    rows.append({
        'id': field('id'),
        'level': field('geographic_level'),
        'years': field('years_available'),
        'access': field('access_method'),
        'format': field('data_format'),
    })

pd.DataFrame(rows)

,id,level,years,access,format
0,bgs_bayern,bavaria,"[2015, 2022]",pdf_or_lgl_agreement,pdf
1,bzga_drogenaffinitaet,germany,"[1973, 1976, 1979, 1982, 1986, 1990, 1993, 199...",gesis_agreement,spss
2,bzga_rauchverhalten,germany,"[1997, 2003, 2007, 2010, 2012, 2013, 2014, 201...",direct_download,pdf
3,debra,germany,,pdf_or_dua,pdf
4,destatis_mikrozensus,germany,"[1999, 2003, 2005, 2009, 2013, 2017]",direct_scrape,html_table
5,eurobarometer_tobacco,germany,"[2003, 2006, 2009, 2012, 2017, 2021]",gesis_download,spss
6,gesundheitsatlas_bayern,bavaria,"[2015, 2022]",manual_export,excel
7,itc_germany,germany,"[2016, 2018, 2020, 2022]",itc_agreement,spss
8,muenchen_gesundheitsbefragung,munich,"[2016, 2021]",pdf_or_negotiation,pdf
9,muenchen_gesundheitsbericht,munich,"[2004, 2009, 2014, 2020]",direct_download,pdf
